> 在线讲义：[第 3 章 · 执行器：电机 + 减速器 = 一个关节](https://wikieden.github.io/robot-architecture/lessons/03-actuators.html) · 本书《机器人软硬件架构》

# 第 3 章 · 执行器：电机 + 减速器 = 一个关节

配套讲义：仓库里 `robot-system-architecture/notes/lessons/03-actuators.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（电机 K_t = 0.1 N·m/A、转子惯量 1e-4 kg·m²、4000 rpm、效率 0.9 都是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 电流即力矩 | $\tau_m = K_t i$ | `motor_torque(i)` | 10 A → 1 N·m |
| 第 2 步 四个后果 | $\eta N\tau_m$、$\omega_m/N$、$N^2J_m$、∝ N | `gear(n, tau_m, rpm_m)` | N = 160：144 N·m、25 rpm、2.56 kg·m² |
| 第 2 步 N = 100 | $N^2$ | `100**2` | 反射惯量放大 10000 倍 |
| 数学原理 能量守恒 | $\tau_m\omega_m\eta = \tau_{out}\omega_{out}$ | `power_balance` | 两边都是 376.99 W |
| 第 4 步 SEA | $\tau = k(\theta_m/N - \theta_{joint})$ | `sea_torque` | 300 N·m/rad × 0.01 rad = 3 N·m |
| 第 5 步 选型 | $mgr + J\alpha$、$i^2R$、转速 × N | `peak_torque`、`heat` | 7.886 N·m；电流翻倍发热 4 倍；N = 160 时转速不够 |

电机参数是编的（K_t = 0.1 N·m/A、转子惯量 1e-4 kg·m²、4000 rpm），不是练习题里那台电机。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
KT = 0.1  # 力矩常数 K_t（N·m/A），编的
J_ROTOR = 1e-4  # 转子惯量 J_m（kg·m²），编的
MOTOR_RPM = 4000.0  # 电机最高转速（rpm），编的
ETA = 0.9  # 减速器效率 η，编的

**函数 `motor_torque`**：电机力矩 τ_m = K_t · i。

In [ ]:
def motor_torque(i_amp: float) -> float:
    """电机力矩 τ_m = K_t · i。"""
    return KT * i_amp

**函数 `gear`**：减速比 N 的四个后果：力矩 ×ηN、转速 ÷N、反射惯量 ×N²、反驱力矩大致 ×N。

In [ ]:
def gear(n: float, tau_m: float, rpm_m: float, eta: float = ETA) -> dict[str, float]:
    """减速比 N 的四个后果：力矩 ×ηN、转速 ÷N、反射惯量 ×N²、反驱力矩大致 ×N。"""
    return {
        "tau_out": eta * n * tau_m,
        "rpm_out": rpm_m / n,
        "j_refl": n**2 * J_ROTOR,
        "backdrive_rel": n,  # 相对 N = 1 的倍数（摩擦 × N 的粗略模型）
    }

**函数 `power_balance`**：能量守恒：输入功率 × η = 输出功率，返回 (τ_m ω_m η, τ_out ω_out)。

In [ ]:
def power_balance(n: float, tau_m: float, omega_m: float, eta: float = ETA) -> tuple[float, float]:
    """能量守恒：输入功率 × η = 输出功率，返回 (τ_m ω_m η, τ_out ω_out)。"""
    g = gear(n, tau_m, 0.0, eta)
    return tau_m * omega_m * eta, g["tau_out"] * (omega_m / n)

**函数 `sea_torque`**：串联弹性执行器：τ = k (θ_电机侧 − θ_关节侧)，θ_电机侧 已经换算到减速器输出端。

In [ ]:
def sea_torque(k: float, theta_motor_side: float, theta_joint: float) -> float:
    """串联弹性执行器：τ = k (θ_电机侧 − θ_关节侧)，θ_电机侧 已经换算到减速器输出端。"""
    return k * (theta_motor_side - theta_joint)

**函数 `peak_torque`**：选型峰值力矩 τ_peak ≈ m g r + J α。

In [ ]:
def peak_torque(m: float, r: float, inertia: float, alpha: float, g: float = 9.81) -> float:
    """选型峰值力矩 τ_peak ≈ m g r + J α。"""
    return m * g * r + inertia * alpha

**函数 `heat`**：绕组发热 P = i² R。

In [ ]:
def heat(i_amp: float, r_ohm: float) -> float:
    """绕组发热 P = i² R。"""
    return i_amp**2 * r_ohm

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：电机力矩和电流成正比 τ_m = K_t i")
for i in (5.0, 10.0):
    print(f"  i = {i:>4} A → τ_m = {motor_torque(i):.2f} N·m")

In [ ]:
print("\n第二步：减速比 N 的四个后果（电机 τ_m = 1 N·m、4000 rpm）")
print(f"  {'N':>5} {'输出力矩 N·m':>12} {'输出转速 rpm':>12} {'反射惯量 kg·m²':>16} {'反驱 ×':>7}")
for n in (1, 6, 50, 160):
    g = gear(n, 1.0, MOTOR_RPM)
    print(f"  {n:>5} {g['tau_out']:>12.1f} {g['rpm_out']:>12.1f} {g['j_refl']:>16.4f} {g['backdrive_rel']:>7.0f}")
print(f"  N = 100 时反射惯量放大 N² = {100**2} 倍")

In [ ]:
print("\n第二步：能量守恒 τ_m ω_m η = τ_out ω_out（N = 50）")
omega_m = MOTOR_RPM * 2 * np.pi / 60
pin, pout = power_balance(50, 1.0, omega_m)
print(f"  输入 × η = {pin:.2f} W，输出 = {pout:.2f} W")

In [ ]:
print("\n第四步：SEA 用弹簧测力 τ = k (θ_电机侧 − θ_关节侧)")
print(f"  k = 300 N·m/rad，两边差 0.01 rad → τ = {sea_torque(300.0, 0.51, 0.50):.2f} N·m")

In [ ]:
print("\n第五步：选型三个数（编的：2 kg 小臂、质心 0.3 m、J = 0.2 kg·m²、α = 10 rad/s²）")
print(f"  峰值力矩 τ_peak ≈ m g r + J α = {peak_torque(2.0, 0.3, 0.2, 10.0):.3f} N·m")
print(f"  发热 i² R：10 A × 0.5 Ω → {heat(10, 0.5):.0f} W；电流翻倍 → {heat(20, 0.5):.0f} W（4 倍）")
need_rpm = 30.0
for n in (50, 160):
    ok = need_rpm * n <= MOTOR_RPM
    print(f"  关节要 {need_rpm} rpm，N = {n} → 电机要 {need_rpm * n:.0f} rpm，{'够' if ok else '不够'}（额定 {MOTOR_RPM:.0f}）")

## 3. 代码里才看得到的

- 能量守恒要用角速度 rad/s 而不是 rpm：`omega_m = rpm × 2π / 60`，否则两边功率对不上。
- `sea_torque` 的"电机侧角度"要先除以 N 换算到减速器输出端，和讲义公式 $\theta_m/N$ 一致；代码里直接传换算后的值。

## 4. 自己改一改

- 在表里加一个 N = 30，看它落在哪种减速器的区间（对照动画的背景色带）。
- 把 `ETA` 改成 0.7（谐波减速器效率偏低的情况），看输出力矩和能量守恒怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 电机快而弱，减速器换成慢而强；力矩乘 N，速度除 N，惯量乘 N 平方。
> 撞地多的腿选低减速比，要精度的臂选谐波，要大推力的选丝杠。